# Aufgabe 3_2_1 – Konstantlicht-Regelung (geschlossener Regelkreis)

## Ziel
Modellieren Sie einen **geschlossenen Regelkreis** für die Konstantlichtregelung.

## Modell-Aufbau

```
Step(w=100) ──▶ [Sum] ──▶ Gain(Kp) ──▶ TransportDelay(0.5) ──▶ PT1(τ=2) ──▶ y
                 ↑                                                             │
                 └─────────────────────── Rückführung ────────────────────────┘
```

| Block | Klasse | Parameter |
|-------|--------|-----------|
| Sollwert | `Step` | `final=100` (Lux) |
| Summationsstelle | `Sum` | $e = w - y_m$ (spät verbinden) |
| Regler | `Gain` | `K=1` (zunächst) |
| Totzeit | `TransportDelay` | `Tt=0.5` |
| Strecke | `PT1` | `tau=2`, `K=1` |
| Rückführung | direkter Pfad | — |

> **Wichtig:** `Sum()` wird zuerst erzeugt, aber erst nach der Strecke verbunden – das ermöglicht die Rückführung.

In [ ]:
import sys
sys.path.insert(0, '..')
from blockdiagram import Step, Sum, Gain, TransportDelay, PT1, Scope

In [ ]:
# Regelkreis aufbauen
Kp = 1    # Regler-Verstärkung

w       = Step(final=100)          # Sollwert: 100 Lux
e       = Sum()                    # Summationsstelle (Eingänge folgen)
regler  = Gain(K=Kp, source=e)
delay   = TransportDelay(Tt=0.5, source=regler)
strecke = PT1(tau=2, K=1, source=delay)

# Rückführung verbinden
e.connect(w,       sign=+1)   # Sollwert addieren
e.connect(strecke, sign=-1)   # Istwert subtrahieren

Scope(t_end=20, ylabel="Helligkeit [Lux]",
      title=f"Konstantlichtregelung – Kp = {Kp}").run(
    Sollwert_w=w,
    Istwert_y=strecke,
    Regelabweichung_e=e
)

## Aufgaben

**a)** Welche Komponenten aus dem RA-Schema (Kap. 1.2) werden durch welche Blöcke dargestellt?

*Antwort:*

**b)** Variieren Sie `Kp` (z.B. 0,5 / 2 / 10). Wann wird das System instabil?

| $K_P$ | Verhalten |
|-------|----------|
| 0.5 | ? |
| 2   | ? |
| 10  | ? |

In [ ]:
# Parameterstudie Kp
for Kp in [0.5, 2, 10]:
    w       = Step(final=100)
    e       = Sum()
    regler  = Gain(K=Kp, source=e)
    delay   = TransportDelay(Tt=0.5, source=regler)
    strecke = PT1(tau=2, K=1, source=delay)
    e.connect(w, +1)
    e.connect(strecke, -1)

    Scope(t_end=20, title=f"Kp = {Kp}").run(
        Sollwert=w, Istwert=strecke
    )